In [3]:
from aiomoex import get_board_candles
import asyncio
import aiohttp
from datetime import datetime, timedelta
import pandas as pd


VALID_INTERVALS = {
        1: "1 минута",
        10: "10 минут", 
        60: "1 час",
        24: "1 день",
        7: "1 неделя",
        31: "1 месяц"
    }

async def fetch_ticker_data(session: aiohttp.ClientSession, ticker: str, interval: int, start_date: str, end_date: str) -> dict[str, list[dict[str, str | int | float]]]:
    try:
        res = await get_board_candles(session, ticker, interval, start_date, end_date)
        return {ticker: res}
    except Exception as e:
        print(f'Ошибка парсинга. Не удалось получить данные для {ticker}, {e}')
        return {ticker: []}

async def get_moex_data(tickers: list[str], days: int = 180, interval: int = 24):
    if interval not in VALID_INTERVALS:
        raise ValueError(f"Неверный интервал. Допустимые значения: {list(VALID_INTERVALS.keys())}")
    
    end_date = datetime.now()
    start_date = (end_date - timedelta(days=days)).strftime('%Y-%m-%d')
    end_date = end_date.strftime('%Y-%m-%d')
    

    async with aiohttp.ClientSession() as session:
        coros = [fetch_ticker_data(session, ticker, interval, start_date, end_date) for ticker in tickers]
        stock_data = await asyncio.gather(*coros)
    stock_data = {k: v for d in stock_data for k, v in d.items()}
    return stock_data


tickers = ['SBER', 'GAZP', 'LKOH', 'ROSN', 'YDEX', 'VTBR', 'TATN', 'GMKN']

stock_data = await get_moex_data(tickers, interval=31)

sber_df = pd.DataFrame(stock_data['SBER'])
sber_df['ticker'] = 'SBER'
print(sber_df)

     open   close    high     low         value     volume  \
0  307.80  308.74  313.71  291.66  2.074161e+11  682166570   
1  308.69  316.15  325.43  300.51  1.908521e+11  608807870   
2  316.69  304.95  329.23  300.25  2.766995e+11  884199380   
3  305.41  310.99  322.90  302.89  1.820776e+11  587813609   
4  310.86  288.36  314.25  285.24  1.705371e+11  568036667   
5  288.88  286.81  295.87  278.00  9.743319e+10  340096853   

                 begin                  end ticker  
0  2025-05-01 00:00:00  2025-05-31 00:00:00   SBER  
1  2025-06-01 00:00:00  2025-06-30 00:00:00   SBER  
2  2025-07-01 00:00:00  2025-07-31 00:00:00   SBER  
3  2025-08-01 00:00:00  2025-08-31 00:00:00   SBER  
4  2025-09-01 00:00:00  2025-09-30 00:00:00   SBER  
5  2025-10-01 00:00:00  2025-10-31 00:00:00   SBER  
